# Module 7 Assignment: Rebuild Your Persona Agent on the OpenAI Agents SDK

Lab 6 rebuilt Study Buddy on top of the OpenAI Agents SDK: same persona, same notes, same three
tools from Lab 5, but with `Agent` and `Runner` replacing the hand written loop and dispatcher.

This assignment makes you do the same move on your **own** Persona Agent. By the end of Module 6
your agent had a persona, study notes, and three tools routed by a `call_tool(name, arguments)`
dispatcher you wrote by hand. Nothing about what your agent *does* needs to change here: you are
porting it to the framework, not redesigning it.

**What you'll build:**
1. An Azure configured `OpenAIChatCompletionsModel`, with tracing disabled, exactly like Lab 6
2. Your persona and notes, carried over from your Module 6 assignment
3. All three of your existing tools, each ported to `@function_tool`
4. One `Agent` that declares your persona, your notes, and all three tools
5. A test using `await Runner.run`
6. The same Gradio chat interface every lab and assignment has used since Lab 4, adapted from Lab 6

**Provider:** Azure OpenAI via APIM, deployment `gpt-5.4-ptu`, same as every lab and assignment so
far.

If you get stuck on any step, look at how Lab 6 made the same kind of change to each of Study
Buddy's three tools, and do the analogous thing to yours.

## 0. Install the packages (run once)

Same as Lab 6.

In [2]:
!uv pip install --upgrade openai openai-agents pypdf python-dotenv requests gradio

Resolved 79 packages in 373ms
Checked 79 packages in 6ms


## 1. Imports

**TODO:** Copy the imports from Lab 6, Step 1 (including `gradio as gr`), plus `requests` (and
anything else your own tools from the Module 6 assignment need, for example `html` if one of your
tools calls an API that returns HTML-escaped text).

In [3]:
# TODO: copy your imports from Lab 6 (including gradio as gr), plus requests and anything your tools need
from dotenv import load_dotenv
import os
import asyncio
import html
from typing import Literal, Optional

import requests
from openai import AsyncAzureOpenAI
from agents import Agent, Runner, function_tool, OpenAIChatCompletionsModel, set_tracing_disabled
from pypdf import PdfReader
import gradio as gr

## 2. Load your Azure OpenAI credentials from `.env`

Identical to Lab 6 and every assignment before it.

In [4]:
from dotenv import load_dotenv
import os

load_dotenv(override=True)

AZURE_OPENAI_ENDPOINT = os.getenv("AZURE_OPENAI_ENDPOINT")
AZURE_OPENAI_API_KEY = os.getenv("AZURE_OPENAI_API_KEY")
DEPLOYMENT = "gpt-5.4-ptu"

print("Endpoint loaded:", bool(AZURE_OPENAI_ENDPOINT))
print("API key loaded:", bool(AZURE_OPENAI_API_KEY))

Endpoint loaded: True
API key loaded: True


## 3. Configure the OpenAI Agents SDK for Azure OpenAI

**TODO:** Copy this setup from Lab 6, Step 3, unchanged: call `set_tracing_disabled(True)`, build an
`AsyncAzureOpenAI` client from your endpoint, key, and API version, then wrap it and your deployment
name in `OpenAIChatCompletionsModel`.

In [5]:
# TODO: set_tracing_disabled(True), then build azure_client and azure_model, exactly like Lab 6
set_tracing_disabled(True)

azure_client = AsyncAzureOpenAI(
    azure_endpoint= AZURE_OPENAI_ENDPOINT,
    api_key=AZURE_OPENAI_API_KEY,
    api_version="2024-10-21",
)

azure_model = OpenAIChatCompletionsModel(
    model=DEPLOYMENT,
    openai_client=azure_client,
)

print("Azure model configured for the OpenAI Agents SDK.")

Azure model configured for the OpenAI Agents SDK.


## 4. Your persona and notes

**TODO:** Bring over your persona description and your study notes from your Module 6 assignment,
unchanged. If your notes lived in a PDF, reuse the same `load_notes` helper from Lab 6, Step 4.

In [6]:
# TODO: persona = "..." and notes = "..." (or load_notes(...) if you used a PDF), from Module 6
persona = (
    "Study Buddy, an upbeat and encouraging AI study coach. Study Buddy keeps quizzes short and "
    "fun, genuinely celebrates every right answer, and never makes a student feel bad about "
    "getting something wrong - just curious to help them learn it."
)

reader = PdfReader("notes/notes.pdf")
notes = ""
for page in reader.pages:
    text = page.extract_text()
    if text:
        notes += text

print(persona)
print(notes[:300], "...")

Study Buddy, an upbeat and encouraging AI study coach. Study Buddy keeps quizzes short and fun, genuinely celebrates every right answer, and never makes a student feel bad about getting something wrong - just curious to help them learn it.
Bloodborne All Bosses Checklist
slaughter all of them
Regular Bosses
Cleric Beast
Father Gascoigne
Blood Starved Beast
Vicar Amelia
Witch of Hemwitch
Shadow of Yharman
Rom Vacuos Spider
Martyr Logarius
Darkbeast Paarl
The One Unborn
Celestial Emissary
Ebrietas
Amygdala
Micolash
Made with ChecklistMa ...


## 5. Port your first tool

**TODO:** Take your first tool from the Module 6 assignment and rewrite it as a plain, typed Python
function decorated with `@function_tool`, the same way Lab 6, Step 5a ported
`log_missed_question_tool`. Keep the function's actual behavior identical: only the decorator and
the removal of your hand written JSON schema should change.

In [7]:
# TODO: your first tool, decorated with @function_tool
LOG_PATH = "missed_questions.txt"

@function_tool
def log_missed_question_tool(question: str) -> str:
    """Record a study question the student got wrong or was unsure about, so they can review it later.

    Args:
        question: The exact question the student missed or was unsure about.
    """
    print(f"Tool called to log a missed question: {question}")
    with open(LOG_PATH, "a", encoding="utf-8") as f:
        f.write(question + "\n")
    return "Missed question logged"

print("Tool ready:", log_missed_question_tool.name)

Tool ready: log_missed_question_tool


## 6. Port your second tool

**TODO:** Same idea, for your second tool (in Module 6 this was the one backed by a free, no-key
external API), the same way Lab 6, Step 5b ported `fetch_bonus_question_tool`.

In [10]:
# TODO: your second tool, decorated with @function_tool
OPENTDB_CATEGORIES = {"general_knowledge": 9, "science_computers": 18}

@function_tool
def fetch_bonus_question_tool(category: Optional[Literal["general_knowledge", "science_computers"]] = None) -> str:
    """Fetch a fun bonus trivia question from an external trivia database, for when the student asks for a bonus question or a break from the study notes.

    Args:
        category: Which trivia category to pull a bonus question from, if the student asked for one.
    """
    print(f"Tool called to fetch a bonus trivia question (category={category})")
    params = {"amount": 1, "type": "multiple"}
    if category in OPENTDB_CATEGORIES:
        params["category"] = OPENTDB_CATEGORIES[category]

    try:
        response = requests.get("https://opentdb.com/api.php", params=params, timeout=20)
        data = response.json()
    except (requests.exceptions.RequestException, ValueError) as e:
        print(f"fetch_bonus_question_tool network error: {e}")
        return "The trivia API is temporarily unavailable - just skip the bonus question and continue the quiz."

    if data.get("response_code") != 0 or not data.get("results"):
        return "Could not fetch a bonus question right now - just skip it and continue the quiz."

    q = data["results"][0]
    question = html.unescape(q["question"])
    answer = html.unescape(q["correct_answer"])
    return f"Bonus question: {question} (correct answer: {answer})"

print("Tool ready:", fetch_bonus_question_tool.name)

Tool ready: fetch_bonus_question_tool


## 7. Port your third tool

**TODO:** Same idea, for your third, self-designed tool, the same way Lab 6, Step 5c ported
`check_progress_tool`.

In [11]:
# TODO: your third tool, decorated with @function_tool
@function_tool
def check_progress_tool() -> str:
    """Check how many study questions the student has missed so far, for when they ask about their progress."""
    print("Tool called to check quiz progress")
    try:
        with open(LOG_PATH, encoding="utf-8") as f:
            missed = [line for line in f if line.strip()]
    except FileNotFoundError:
        missed = []

    if not missed:
        return "No questions have been missed yet - a clean sheet so far!"
    return f"So far, {len(missed)} question(s) have been logged as missed."

print("Tool ready:", check_progress_tool.name)

Tool ready: check_progress_tool


## 8. Build your Agent

**TODO:** Build one `Agent`, the same way Lab 6, Step 6 built Study Buddy: a `name`, `instructions`
built from your persona and your notes (with the same rules your old system prompt had about when
to use each tool), a `tools` list containing all three of your ported tools, and `model=azure_model`.

Notice what is missing compared to your Module 6 assignment: no `call_tool(name, arguments)`
dispatcher, and no JSON schema for any of the three tools. `Agent` and `@function_tool` already
cover both.

In [12]:
# TODO: instructions = f"""..."""  (persona + notes + rules, like Module 6's system prompt)
# TODO: your_agent = Agent(name=..., instructions=instructions, tools=[...], model=azure_model)
instructions = f"""
# Your role

You are {persona}

You quiz a student on the study notes below.

# Context

Here are the study notes you are quizzing the student on the order of:

{notes}

# Rules

Ask exactly ONE question at a time, based only on the notes above. Wait for the student's answer
before asking the next question.

After the student answers, say clearly whether they were right or wrong, and give a one-sentence
explanation either way, staying in character the whole time.

The study notes are a list of all boss type enemies from the 2015 video game Bloodborne by FromSoftware. 
The objective of the study session is to memorize all bosses in order. The names can be counted as correct 
as long as they contain most of the boss' name spelled correctly, or at least one word from the objective name.

If the student's answer was wrong, or they say "I don't know", you must record it as a missed
question using log_missed_question_tool before moving on to the next question.

If the student asks for a bonus question, or seems to want a break from the notes, use
fetch_bonus_question_tool to get one - this question does not need to relate to the notes.

If the student asks how they are doing, or asks for a progress check, use check_progress_tool to
tell them how many questions they've missed so far.

IMPORTANT: only ask quiz questions from the notes above. If the student asks about something
outside the notes (and it isn't a bonus question or a progress check), say that it isn't covered
in this study set.
"""

study_buddy = Agent(
    name="BB Boss Guide",
    instructions=instructions,
    tools=[log_missed_question_tool, fetch_bonus_question_tool, check_progress_tool],
    model=azure_model,
)

print("Agent built:", study_buddy.name)

Agent built: BB Boss Guide


## 9. Test it: `await Runner.run`

**TODO:** Run your agent with `await Runner.run(your_agent, "...")`, the same way Lab 6, Step 7
tested Study Buddy. Pick a message that should trigger at least one of your three tools, and
confirm the tool's visible effect (a line appended to a file, a number that changed, or whatever
your tool does) actually happened.

Stick to `await Runner.run(...)` here, not `Runner.run_sync`: Jupyter notebooks already run their
own event loop in the background, and `Runner.run_sync` raises
`RuntimeError: AgentRunner.run_sync() cannot be called when an event loop is already running.` when
called from a notebook cell.

In [13]:
# TODO: result = await Runner.run(your_agent, "...")
# TODO: print(result.final_output)
result = await Runner.run(study_buddy, "Hi! Can you quiz me with one question from the bonus question database?")
print(result.final_output)

Tool called to fetch a bonus trivia question (category=None)
You got it — bonus round! 🎉

Bonus question: Which Beatle led the way across the zebra crossing on the Abbey Road album cover?


## 10. Give it a real chat UI with Gradio

**TODO:** Adapt the `chat` function and `gr.ChatInterface` call from Lab 6, Step 8 to your own
agent: an `async def chat(message, history)` function that rebuilds `history` as a plain list of
`{"role": ..., "content": ...}` dictionaries, appends the new `message` as one more `user` turn,
passes that to `await Runner.run(your_agent, ...)`, and returns `result.final_output`, handed to
`gr.ChatInterface(chat).launch(inbrowser=True)`.

As in Lab 6, `gr.ChatInterface` accepts your async `chat` function directly, and its `history`
already comes in the shape `Runner.run` expects, so no extra conversion is needed.

Try a conversation that would trigger more than one of your three tools across a few turns, then
interrupt the cell (stop button) when you are done chatting.

In [14]:
# TODO: async def chat(message, history): ... (see Lab 6, Step 8)
# TODO: gr.ChatInterface(chat).launch(inbrowser=True)

async def chat(message, history):
    history = [{"role": item["role"], "content": item["content"]} for item in history]
    result = await Runner.run(study_buddy, history + [{"role": "user", "content": message}])
    return result.final_output

gr.ChatInterface(chat).launch(inbrowser=True)

* Running on local URL:  http://127.0.0.1:7861
* To create a public link, set `share=True` in `launch()`.


Tool called to fetch a bonus trivia question (category=None)
Tool called to log a missed question: Bonus question: Which two modern-day countries used to be known as the region of Rhodesia between the 1890s and 1980?
Tool called to log a missed question: What is the second regular boss?Tool called to fetch a bonus trivia question (category=None)



## Reflection Questions

Answer these in a new Markdown cell at the end of your notebook:

1. Which parts of your Module 6 assignment did you copy over completely unchanged? Which parts,
   if any, needed a real change rather than just a different decorator?
2. Compare your old `call_tool(name, arguments)` dispatcher to what `Agent` and `Runner` do for you
   now. What exactly is `Runner.run` doing in place of that dispatcher and your `while`/`for` loop?
3. Your Module 6 system prompt included sentences telling the model when to use each tool. Did any
   of that wording need to change for the OpenAI Agents SDK, or did it carry over directly into
   `instructions`?
4. If you added a fourth tool to this agent, what would you need to write, and what would you not
   need to write, compared to adding a fourth tool by hand back in Module 6?

## Submission Checklist

- Remove your `.env` file before submitting; it contains your API key.
- Run the notebook top to bottom and confirm it executes cleanly from a fresh kernel.
- Confirm all three of your ported tools are reachable, `tools=[...]` lists all three, and at least
  one tool call actually happened during your test in Step 9 or Step 10.
- Answer all four reflection questions in a Markdown cell at the end of the notebook.
- Rename the file to include your name, for example `Module7Assignment_JaneDoe.ipynb`, then submit
  it.

1. Which parts of your Module 6 assignment did you copy over completely unchanged? Which parts, if any, needed a real change rather than just a different decorator?
I was able to take the persona of the agent and the notes themselves. As well as a bunos question asker tool which fit in nicely.


2. Compare your old `call_tool(name, arguments)` dispatcher to what `Agent` and `Runner` do for you now. What exactly is `Runner.run` doing in place of that dispatcher and your `while`/`for` loop?
Instead of using the while/for loop to directly call and search for arg and func entries, Runner takes care of actually pathing. A while/for loop needs pathing to be hard coded to be able to know what the code is intended to do, where, and when. Runner instead defines whaty the agent is able to do and let the agent definen how the loop runs, instead of having a pre-set program.


3. Your Module 6 system prompt included sentences telling the model when to use each tool. Did any of that wording need to change for the OpenAI Agents SDK, or did it carry over directly into `instructions`?
For the most part it was able to carry directly over. There were a few instances where the wording changed to directly tell the agent what the name of the tool was for the purpose of calling, but otherwise the rest of the prompt engineering stayed the same.


4. If you added a fourth tool to this agent, what would you need to write, and what would you not need to write, compared to adding a fourth tool by hand back in Module 6?
I wouldn't need to write the JSON schema as the agent takes care of that for me. What I do need to write is the function itself with @function_tool and an update to the tools=[] list for the agent to be able to find/call the tool itself.

